# Gated Linear Attention (GLA) — a toy-scale build

A minimal implementation of **Gated Linear Attention**, from Yang et al.,
*"Gated Linear Attention Transformers with Hardware-Efficient Training"*
(2023) — the direct architectural ancestor of KDA (see the `kda/` folder in
this repo), stripped down to its simplest form.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

GLA is a **linear attention** layer: instead of storing every past token and
attending over all of them (which is what makes normal attention's memory
grow with sequence length), it keeps a fixed-size running **state** — a
small matrix — and updates it one token at a time.

Plain linear attention has one big problem: it only ever *adds* new
information to the state, so old information never fades. GLA's fix is a
**per-channel forget gate**: before writing anything new in, the state gets
multiplied by a decay value between 0 and 1, so old information gradually
fades unless the model chooses to keep it.

At every timestep:

```
alpha_t = sigmoid(W_alpha x_t)            # per-channel forget gate, in (0,1)
S_t = diag(alpha_t) * S_{t-1} + k_t (x) v_t     # decay, then write (outer product)
o_t = q_t^T S_t                            # read out with the query
```

That's the entire mechanism — decay the state, add the new key-value outer
product, read with the query.

**How this compares to KDA:** KDA (in the `kda/` folder) starts from exactly
this same gated-linear-attention idea and adds one more ingredient — the
**delta rule**. Where GLA just adds the new key-value pair on top of the
decayed state, KDA first *erases* whatever was previously written for a
similar key before writing the new value. GLA is what you get with that
erase step removed — simpler, and a good place to see the "forget gate on a
matrix state" idea in isolation before layering more on top of it.

> **Simplification used here:** the real implementation computes this with
> a **chunkwise-parallel** algorithm for GPU efficiency. This notebook uses
> the plain sequential recurrence (a Python loop over timesteps) — same
> math, much easier to read.

In [ ]:
class ShortConv(nn.Module):
    def __init__(self, dim, kernel_size=4):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(dim, dim, kernel_size, groups=dim, padding=0)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))
        return self.conv(x).transpose(1, 2)

In [ ]:
class GLA(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32, conv_kernel=4):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.q_conv = ShortConv(inner, conv_kernel)
        self.k_conv = ShortConv(inner, conv_kernel)
        self.alpha_proj = nn.Linear(d_model, inner, bias=True)   # per-channel forget gate
        self.gate_proj = nn.Linear(d_model, inner, bias=True)    # output gate
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.out_norm = nn.LayerNorm(d_head)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = F.silu(self.q_conv(self.q_proj(x))).view(B, T, H, Dh)
        k = F.silu(self.k_conv(self.k_proj(x))).view(B, T, H, Dh)
        v = self.v_proj(x).view(B, T, H, Dh)
        alpha = torch.sigmoid(self.alpha_proj(x)).view(B, T, H, Dh)   # decay, per channel

        S = x.new_zeros(B, H, Dh, Dh)
        outs = []
        for t in range(T):
            a_t, k_t, v_t, q_t = alpha[:, t], k[:, t], v[:, t], q[:, t]
            S = a_t.unsqueeze(-1) * S + k_t.unsqueeze(-1) * v_t.unsqueeze(-2)   # decay, then write
            o_t = torch.einsum('bhd,bhde->bhe', q_t, S)                         # read
            outs.append(o_t)

        o = self.out_norm(torch.stack(outs, dim=1)).reshape(B, T, H * Dh)
        gate = torch.sigmoid(self.gate_proj(x))
        return self.out_proj(gate * o)

## 2. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([GLA(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through GLA once it's wired into a real model. So the rest of this
notebook:

1. wraps GLA into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Add the delta rule** (the erase-then-write step) and you've built KDA —
  see the `kda/` folder for the full version with that extra ingredient.
- **Compare against RetNet** (`retnet/` folder) — same outer-product-write
  state update, but with a *fixed* per-head decay instead of GLA's
  input-dependent one. Training both on the same toy task is a good way to
  feel the difference a *learned* forget gate makes.
- **Switch to the chunkwise-parallel form** for real speed — the sequential
  loop here is for readability, not performance.

Reference: Yang et al., *"Gated Linear Attention Transformers with
Hardware-Efficient Training,"* 2023.